### RAG Pipeline -> Data ingestion tp Vector dB Pipeline

In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader


### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Processes all PDF files in the specified directory and returns a list of documents."""

    all_documents = []

    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory '{pdf_directory}'.")

    for pdf_file in pdf_files:

        print(f"\nProcessing file: {pdf_file}")

        try:
            # Load PDF
            loader = PyMuPDFLoader(str(pdf_file))

            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Successfully processed: {len(documents)} pages.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents processed: {len(all_documents)}")

    return all_documents


# Process all PDFs
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files in the directory '../data'.

Processing file: ..\data\pdf\Front End Developer - Preparation Document-1 (1).pdf
Successfully processed: 3 pages.

Processing file: ..\data\pdf\Software Developer - Preparation Document.pdf
Successfully processed: 3 pages.

Total documents processed: 6


In [20]:

## Text Splitting into Chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Splits documents into smaller chunks using
    RecursiveCharacterTextSplitter.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    all_chunks = []

    for doc in documents:

        # Split document text
        chunks = text_splitter.split_text(doc.page_content)

        # Create Document chunk objects
        for i, chunk in enumerate(chunks):

            chunk_doc = Document(
                page_content=chunk,
                metadata={
                    **doc.metadata,
                    "chunk_index": i
                }
            )

            all_chunks.append(chunk_doc)

    print(f"Total chunks created: {len(all_chunks)}")

    return all_chunks

In [19]:
chunks=split_documents(all_pdf_documents)
chunks

Total chunks created: 14


[Document(metadata={'producer': 'Skia/PDF m102 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'file_path': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'Front End Developer - Preparation Document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'Front End Developer - Preparation Document-1 (1).pdf', 'file_type': 'pdf', 'chunk_index': 0}, page_content="Front End Developer - Preparation Document\nDear Students,\nWe hire tech enthusiasts with a broad set of technical skills who are ready to tackle\nsome of technology's greatest challenges. The hiring process has been designed from\nthe ground level to avoid any false positives and in order to help you to get through\nour process.\nWe have curated this document after examining some of the 

### Embedding and VectorStoreDB